In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib_inline
import re
import os
import itertools
import matplotlib.ticker as ticker
import matplotlib.transforms as mtransforms
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
abspath=os.getcwd()

In [ ]:
# helper functions

def extract_data(file):
    # reads a .stat file with tab-separated columns; first row is header
    with open(file, 'r') as f:
        lines = f.read().splitlines()
        info_ = lines[0].strip().split('	')
        info = [" ", info_[0], info_[1]]

    data = {}
    for i in range(1, len(info)):
        temp = []
        for j in range(1, len(lines)):
            line = lines[j].strip()
            split_line = line.split('	')
            if len(split_line) > i:
                temp.append(float(split_line[i]))
            else:
                print(f"Warning: Skipping line {j} due to insufficient columns: {line}")
        data[info[i]] = temp
    return info, data

def extract_data_cflp(file):
    with open(file, 'r') as f:
        lines = f.read().splitlines()
    info = re.split(r'\s+', lines[0])

    data = {}
    for i in range(1, len(info)):
        temp = []
        for j in range(1, len(lines)):
            temp.append(float(lines[j].split()[i]))
        data[info[i]] = temp
    return info, data


def process_data(data, arg):
    # builds (x, y) for a performance profile curve
    data = sorted(data)
    x = []
    y = []

    if "node" in arg:
        x.append(1)
        for i in range(len(data)):
            if data[i] < 1000000 - 0.01 and data[i] > 1 + 0.01:
                if len(x) == 1:
                    y.append((i + 1) / len(data))
                x.append(data[i])
                y.append((i + 1) / len(data))
        x.append(1000000)
        y.append(y[-1])

    elif "time" in arg:
        x.append(0)
        for i in range(len(data)):
            if data[i] < 7200 - 0.01 and data[i] > 1 + 0.01:
                if len(x) == 1:
                    y.append((i + 1) / len(data))
                x.append(data[i])
                y.append((i + 1) / len(data))
        x.append(7200)
        y.append(y[-1])

    elif "endgap" in arg:
        x.append(0)
        for i in range(len(data)):
            if data[i] < 100 - 0.01:
                if len(x) == 1:
                    y.append((i + 1) / len(data))
                x.append(data[i])
                y.append((i + 1) / len(data))
        x.append(100)
        y.append(y[-1])

    return x, y


def get_xlabel(str):
    if "time" in str:    return "CPU time [s]"
    elif "node" in str:  return "# Nodes"
    elif str == "rgap":  return "Root gap [%]"
    elif "endgap" in str: return "Gap [%]"
    elif str == "UB":    return "UB [%]"
    elif str == "LB":    return "LB [%]"
    return ""


In [ ]:
# plot settings
markers = ['s', 'o', '*', '*', '*', ' ', 'x']
colors  = ['b', 'r', 'g', 'c', 'm', 'y', 'k', '#FF0000']


In [ ]:
is_show_end_value = False

dir_data = abspath + '/stat/'
dir_fig  = abspath + '/eps'
dir_png  = abspath + '/png'

if not os.path.exists(dir_fig):
    os.mkdir(dir_fig)
if not os.path.exists(dir_png):
    os.mkdir(dir_png)

args = [
    'time_m_500', 'time_m_800', 'time_m_1000',
    'time_n_500', 'time_n_800', 'time_n_1000',
    'time_p_2',   'time_p_5',   'time_p_10',  'time_p_50',
    'time_r_2',   'time_r_5',   'time_r_10',  'time_r_50',
]

if not os.path.exists(dir_fig):
    os.mkdir(dir_fig)

for arg in args:
    path = dir_data + arg + ".stat"
    if arg == "node1":
        info, data = extract_data_cflp(path)
    else:
        info, data = extract_data(path)

    save_path = os.path.join(dir_fig, arg)

    markersize = 8
    linewidth  = 1
    fig, ax = plt.subplots(figsize=(8, 4))

    ax.set_ylim(0, 1.02)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1], ['0', '25', '50', '75', '100'], fontsize=15, fontweight='bold')

    ax.set_xscale('log')
    ax.set_xlim(1, 7200)
    ax.set_xticks([1, 10, 100, 1000, 7200], ['1', '10', '100', '1000', '7200'], fontsize=15, fontweight='bold')
    ax.set_xticklabels([1, 10, 100, 1000, 7200], fontsize=15)

    # fix aspect ratio so 1 log-decade = 0.25 on y-axis
    x_len = np.log10(10) - np.log10(1)
    y_len = 0.25
    ax.set_aspect(1 / (y_len / x_len), adjustable='box')

    if "node" in arg:
        ax.set_xlim(1, 1000000)
        ax.set_xscale('log')
    elif "endgap" in arg:
        ax.set_xlim(1e-3, 100)
        ax.set_xscale('log')
        ax.set_xticks([0.001, 0.01, 0.1, 1, 10, 100])
        ax.set_xticklabels(['0.001', '0.01', '0.1', '1', '10', '100'], fontsize=15, fontweight='bold')

    i = 0
    for key in data.keys():
        x, y = process_data(data[key], arg)
        ax.plot(x, y, label=key, markersize=markersize, marker=markers[i],
                color=colors[i], linewidth=linewidth, markerfacecolor='none')
        if "time" in arg and is_show_end_value:
            y_at_7200 = y[-1] * 100
            ax.annotate(f'{y_at_7200:.2f}%',
                xy=(7200, y_at_7200 / 100),
                xytext=(7100, y_at_7200 / 100 - 0.1),
                fontsize=10, color=colors[i])
            plt.tight_layout()
        i += 1

    legend = ax.legend(loc='upper left', prop={'size': 13})
    for text in legend.get_texts():
        text.set_text(text.get_text().strip())
        text.set_weight('bold')

    plt.rcParams['xtick.direction'] = 'in'
    plt.rcParams['ytick.direction'] = 'in'

    ax.set_ylabel('# Instances  [%]', fontsize=15, fontweight='bold')
    ax.set_xlabel(get_xlabel(arg), fontsize=15, fontweight='bold')

    ax.grid(True, axis='x', linestyle='--', linewidth=0.5)
    ax.grid(True, which='minor', linestyle='dashed', alpha=0)
    ax.grid(True, which='major', alpha=1)

    print(save_path)
    plt.savefig(save_path + ".eps", format="eps", bbox_inches='tight')
    plt.savefig(os.path.join(dir_png, arg + ".png"), format="png", dpi=150, bbox_inches='tight')
